# Milestone 2 — RAG Pipeline

Builds the full RAG pipeline incrementally:
- **Step 1**: LLM pipeline (Groq / Llama 3.1)
- **Step 2**: Semantic RAG with FAISS vectorstore + LCEL chain
- **Step 3**: Hybrid RAG (BM25 + Semantic via RRF)

## Setup

Prerequisites:
1. Copy `.env.example` to `.env` and add your `GROQ_API_KEY`
2. Build retrieval artifacts:
   ```bash
   python src/scripts/build_bm25_index.py
   python src/scripts/build_semantic_index.py
   ```
3. Install dependencies: `pip install langchain langchain-core langchain-groq langchain-community langchain-huggingface`

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Project root:", repo_root)

---

## Step 1 — LLM Pipeline (Groq / Llama 3.1)

### 1.1 Initialise the LLM

In [ ]:
from src.rag_pipeline import LLMPipeline

llm_pipeline = LLMPipeline(model="llama-3.1-8b-instant", temperature=0.0, max_tokens=512)
print(f"Pipeline ready — model: {llm_pipeline.model}")

### 1.2 Plain generation (no retrieval)

In [ ]:
query = "What makes a good lip balm?"
answer = llm_pipeline.generate(query)
print(f"Query : {query}")
print("-" * 60)
print(f"Answer: {answer}")

### 1.3 RAG generation (with hand-crafted docs)

In [ ]:
sample_docs = [
    {"title": "Great for dry skin", "text": "Kept my skin hydrated all day without feeling greasy.", "rating": 5.0},
    {"title": "Decent but pricey", "text": "Good for dry skin but a bit greasy at first.", "rating": 4.0},
    {"title": "Did not work for me", "text": "Made my skin break out. Not for sensitive skin.", "rating": 2.0},
]

rag_answer = llm_pipeline.generate("Is this good for dry skin?", documents=sample_docs)
print(f"Answer: {rag_answer}")

### 1.4 Grounding check

In [ ]:
off_docs = [{"title": "Nice hairbrush", "text": "Detangles hair easily.", "rating": 5.0}]
grounding = llm_pipeline.generate("Does this help with acne?", documents=off_docs)
print(f"Answer: {grounding}")
print("\nExpected: model should say the reviews do not contain enough information.")

---

## Step 2 — Semantic RAG Pipeline

### 2.1 Load saved documents and build LangChain FAISS vectorstore

In [ ]:
import pickle

SEMANTIC_INDEX = repo_root / "data/processed/semantic_index"

with open(SEMANTIC_INDEX / "semantic_documents.pkl", "rb") as f:
    documents = pickle.load(f)

print(f"Loaded {len(documents):,} documents")
print("Sample:", {k: str(v)[:60] for k, v in documents[0].items()})

In [ ]:
from src.rag_pipeline import build_semantic_vectorstore

vectorstore = build_semantic_vectorstore(documents)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

print("Vectorstore and retriever ready.")

### 2.2 Test context builder

In [ ]:
from src.rag_pipeline import build_context

test_query = "moisturizer for dry skin"
retrieved = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved)} documents for: '{test_query}'\n")
print(build_context(retrieved))

### 2.3 Prompt variant comparison

We test three prompt variants on the same query to choose the best one.

In [ ]:
from langchain_groq import ChatGroq
from src.rag_pipeline import build_rag_chain, PROMPT_VARIANTS
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0, max_tokens=512)

eval_query = "Is this moisturizer good for sensitive skin?"

for variant in PROMPT_VARIANTS:
    chain = build_rag_chain(retriever, llm, prompt_variant=variant)
    answer = chain.invoke(eval_query)
    print(f"=== Variant: {variant} ===")
    print(answer)
    print()

### 2.4 Full semantic RAG pipeline — test queries

In [ ]:
rag_chain = build_rag_chain(retriever, llm, prompt_variant="concise")

test_queries = [
    "What lip balm is good for very dry lips?",
    "Is there a fragrance-free moisturizer for sensitive skin?",
    "something for frizzy hair",
]

for q in test_queries:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-" * 60)

---

## Step 3 — Hybrid RAG (BM25 + Semantic)

### 3.1 Load BM25 and semantic retrievers, build HybridRetriever

In [ ]:
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever
from src.hybrid import HybridRetriever

bm25_retriever = BM25Retriever.load(repo_root / "data/processed/bm25_index")
sem_retriever = SemanticRetriever.load(repo_root / "data/processed/semantic_index")

hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25_retriever,
    semantic_retriever=sem_retriever,
    top_k=5,
)

print("HybridRetriever ready.")

### 3.2 Test hybrid retrieval

In [ ]:
hybrid_query = "fragrance-free moisturizer for sensitive skin"
hybrid_docs = hybrid_retriever.invoke(hybrid_query)

print(f"Retrieved {len(hybrid_docs)} hybrid docs for: '{hybrid_query}'\n")
print(build_context(hybrid_docs))

### 3.3 Full Hybrid RAG pipeline

In [ ]:
hybrid_rag_chain = build_rag_chain(hybrid_retriever, llm, prompt_variant="concise")

for q in test_queries:
    print(f"Q: {q}")
    print(f"A: {hybrid_rag_chain.invoke(q)}")
    print("-" * 60)

### 3.4 Side-by-side comparison: Semantic vs Hybrid RAG

In [ ]:
compare_query = "What lip balm works best for very dry, chapped lips?"

sem_answer = rag_chain.invoke(compare_query)
hyb_answer = hybrid_rag_chain.invoke(compare_query)

print(f"Query: {compare_query}\n")
print("--- Semantic RAG ---")
print(sem_answer)
print()
print("--- Hybrid RAG ---")
print(hyb_answer)

---

## Summary

| Component | Implementation |
|-----------|---------------|
| LLM | `llama-3.1-8b-instant` via Groq API |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` |
| Vectorstore | LangChain FAISS |
| Hybrid fusion | Reciprocal Rank Fusion (RRF, k=60) |
| Prompt default | `concise` variant |
| Chain | LCEL: retriever → build_context → prompt → LLM → StrOutputParser |

See `results/milestone2_discussion.md` for detailed findings on prompt variants and hybrid vs semantic comparison.